[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pflahert/hd-stats-workshop/blob/main/notebooks/lab3.ipynb)

# Lab 3: The Map That Drew Itself
## Dimension Reduction

**Day 2, 10:00–11:00** | Learning Outcomes: LO2, LO3

In this lab you will:
1. Perform PCA on the ALL dataset and interpret the results
2. Evaluate preprocessing choices (centering, scaling)
3. Apply t-SNE and UMAP — and see them lie
4. Test parameter sensitivity
5. Compare methods and articulate what each can and cannot tell you

The key question: **can we see the B-cell vs. T-cell distinction in lower dimensions?**

In [ ]:
# Setup
!pip install -q umap-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import umap
import warnings
warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('default')

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12

In [ ]:
# Load ALL data
# Try GitHub URL first; if the repo is private, upload the CSVs manually
url_base = "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/main/data/"
try:
    expr = pd.read_csv(url_base + "all_expression.csv", index_col=0)
    meta = pd.read_csv(url_base + "all_metadata.csv", dtype={'sample_id': str})
    print("Loaded data from GitHub.")
except Exception:
    print("Could not fetch from GitHub (repo may be private).")
    print("Upload all_expression.csv and all_metadata.csv when prompted.")
    from google.colab import files
    uploaded = files.upload()
    expr = pd.read_csv("all_expression.csv", index_col=0)
    meta = pd.read_csv("all_metadata.csv", dtype={'sample_id': str})
    print("Loaded data from uploaded files.")

# Ensure column names are strings for safe indexing
expr.columns = expr.columns.astype(str)

# Verify alignment
assert set(meta['sample_id']) == set(expr.columns), \
    "Sample IDs in metadata do not match expression matrix columns!"

# Transpose: expression is genes x samples, we need samples x genes for sklearn
X = expr.T.values  # (128 samples, 12625 genes)
gene_names = expr.index.values

# Use the pre-computed 'subtype' column (consistent with lab1 and lab2)
bt_type = meta['subtype'].values
color_map = {'B': '#2166ac', 'T': '#b2182b'}  # blue for B, red for T
colors = [color_map[t] for t in bt_type]

print(f"Data dimensions (samples x genes): {X.shape}")
print(f"Groups: B={sum(bt_type == 'B')}, T={sum(bt_type == 'T')}")

---

## Part 1: PCA

Perform PCA on the ALL expression matrix. The data should be **centered and scaled** (each gene standardized to mean 0, variance 1). This ensures that genes with larger raw variance do not dominate the first principal components.

We will produce:
1. A **scree plot** showing proportion of variance explained for the first 30 PCs
2. A **cumulative variance** plot — marking where 50% and 80% are reached
3. A **scatter plot** of PC1 vs. PC2, colored by B/T subtype

In [ ]:
np.random.seed(2026)

# Standardize the data: center and scale each gene
scaler = StandardScaler()  # mean=0, std=1 per feature
X_scaled = scaler.fit_transform(X)

# Fit PCA (keep all components up to min(n, p))
pca = PCA(random_state=2026)
pca_scores = pca.fit_transform(X_scaled)

# Variance explained
var_explained = pca.explained_variance_ratio_
cum_var = np.cumsum(var_explained)

# --- Three-panel figure ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Scree plot (first 30 PCs)
ax = axes[0]
ax.bar(range(1, 31), var_explained[:30] * 100, color='steelblue', edgecolor='white')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('Scree Plot')
ax.set_xlim(0.5, 30.5)

# 2. Cumulative variance
ax = axes[1]
ax.plot(range(1, len(cum_var) + 1), cum_var * 100, 'o-', markersize=2, color='steelblue')
# Mark 50% and 80% thresholds
n_50 = np.argmax(cum_var >= 0.50) + 1
n_80 = np.argmax(cum_var >= 0.80) + 1
ax.axhline(50, color='orange', linestyle='--', linewidth=1, label=f'50% at PC{n_50}')
ax.axhline(80, color='red', linestyle='--', linewidth=1, label=f'80% at PC{n_80}')
ax.axvline(n_50, color='orange', linestyle=':', linewidth=1, alpha=0.5)
ax.axvline(n_80, color='red', linestyle=':', linewidth=1, alpha=0.5)
ax.set_xlabel('Number of PCs')
ax.set_ylabel('Cumulative Variance Explained (%)')
ax.set_title('Cumulative Variance')
ax.set_xlim(0, 128)
ax.legend(fontsize=10)

# 3. PC1 vs PC2 scatter
ax = axes[2]
for label in ['B', 'T']:
    mask = bt_type == label
    ax.scatter(pca_scores[mask, 0], pca_scores[mask, 1],
               c=color_map[label], label=label, alpha=0.7, edgecolors='white', s=50)
ax.set_xlabel(f'PC1 ({var_explained[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_explained[1]*100:.1f}%)')
ax.set_title('PC1 vs PC2')
ax.legend(title='Subtype')

plt.tight_layout()
plt.show()

# Report
print(f"PC1 explains {var_explained[0]*100:.2f}% of variance")
print(f"PC2 explains {var_explained[1]*100:.2f}% of variance")
print(f"PCs needed for 50% variance: {n_50}")
print(f"PCs needed for 80% variance: {n_80}")

### Critique Checklist

| Question | Your Answer |
|----------|-------------|
| Did you use `center=True` and `scale=True` (StandardScaler)? | |
| Does PC1 separate B from T? What % variance does it explain? | |
| Is PC1 substantially larger than $1/\min(n,p) \approx 0.78\%$? | |

### Exercise 1.2

With $p = 12{,}625$ and $n = 128$, if all variables were independent noise, each PC would explain about $1/128 \approx 0.78\%$ of variance. Is PC1 substantially larger than this?

**YOUR ANSWER:**

---

## Part 2: Loadings

Extract the loadings for PC1. Which genes drive the separation between B-cell and T-cell ALL?

In [ ]:
# Extract PC1 loadings
pc1_loadings = pca.components_[0]  # shape: (p,)

# Create a Series for easy sorting
loadings_series = pd.Series(pc1_loadings, index=gene_names)

# Top 10 positive and top 10 negative loadings
top_pos = loadings_series.nlargest(10)
top_neg = loadings_series.nsmallest(10)

print("=== Top 10 Positive PC1 Loadings (drive B-cell direction) ===")
print(pd.DataFrame({'Gene': top_pos.index, 'Loading': top_pos.values}).to_string(index=False))
print()
print("=== Top 10 Negative PC1 Loadings (drive T-cell direction) ===")
print(pd.DataFrame({'Gene': top_neg.index, 'Loading': top_neg.values}).to_string(index=False))

# Histogram of all PC1 loadings
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(pc1_loadings, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('PC1 Loading')
ax.set_ylabel('Number of Genes')
ax.set_title('Distribution of PC1 Loadings (12,625 genes)')
plt.tight_layout()
plt.show()

# Note
print(f"\nMost loadings are near zero. Only a subset of genes drive the PC1 separation.")
print(f"Loadings > |0.03|: {np.sum(np.abs(pc1_loadings) > 0.03)} genes")

### Exercise 2.1

Are most loadings near zero? Is the signal in PC1 driven by a few genes or distributed across many?

**YOUR ANSWER:**

---

## Part 3: How Many Components?

How do you decide how many PCs to keep? The scree plot gives a visual heuristic, but we can do better. A **permutation test** compares observed eigenvalues against eigenvalues from shuffled (null) data. PCs whose eigenvalues exceed the null distribution are "significant."

We also compute the **Marchenko-Pastur upper edge**: the largest eigenvalue expected from a pure-noise random matrix.

In [ ]:
np.random.seed(2026)

n_perms = 100  # need enough permutations for a reliable 95th percentile
n_pcs_check = 20
n_samples, n_genes = X_scaled.shape

# Observed eigenvalues (variance explained in units of eigenvalue)
observed_eigenvalues = pca.explained_variance_[:n_pcs_check]

# Permutation: shuffle each column independently, run PCA
perm_eigenvalues = np.zeros((n_perms, n_pcs_check))
for i in range(n_perms):
    X_perm = X_scaled.copy()
    for j in range(X_perm.shape[1]):
        np.random.shuffle(X_perm[:, j])
    pca_perm = PCA(n_components=n_pcs_check, random_state=2026)
    pca_perm.fit(X_perm)
    perm_eigenvalues[i] = pca_perm.explained_variance_

# 95th percentile of permuted eigenvalues
perm_95 = np.percentile(perm_eigenvalues, 95, axis=0)

# Count significant PCs
n_sig_pcs = np.sum(observed_eigenvalues > perm_95)

# Marchenko-Pastur upper edge for the sample covariance matrix
# For standardized data with p >> n, the MP upper edge on the
# eigenvalues of the (1/(n-1)) X^T X matrix is:
#   lambda_+ = (1 + sqrt(p/n))^2
# This will be large (since p/n ~ 99), reflecting that even noise
# eigenvalues are inflated in the p >> n regime.
gamma = n_genes / n_samples  # p/n ratio
mp_upper = (1 + np.sqrt(gamma))**2

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
pcs = np.arange(1, n_pcs_check + 1)
ax.plot(pcs, observed_eigenvalues, 'o-', color='steelblue', linewidth=2,
        markersize=6, label='Observed eigenvalues')
ax.plot(pcs, perm_95, 's--', color='#b2182b', linewidth=2,
        markersize=6, label='95th percentile (permutation)')
ax.axhline(mp_upper, color='orange', linestyle=':', linewidth=2,
           label=f'MP upper edge = {mp_upper:.1f} (p/n = {gamma:.0f})')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Eigenvalue')
ax.set_title('Observed vs. Null Eigenvalues')
ax.legend(fontsize=10)
ax.set_xticks(pcs)
plt.tight_layout()
plt.show()

print(f"Number of significant PCs (above permutation threshold): {n_sig_pcs}")
print(f"Marchenko-Pastur upper edge (p/n = {gamma:.1f}): {mp_upper:.1f}")
print(f"Note: The MP upper edge is very large because p >> n.")
print(f"The permutation test is more informative here.")

---

## Part 4: UMAP

Apply UMAP to the ALL data. **Critical**: run UMAP on the top 30 PCs, not the raw 12,625 dimensions. Two reasons:
1. **Computational**: UMAP on 12,625 features is slow and memory-intensive
2. **Statistical**: most of those 12,625 dimensions are noise — PCA denoises first

In [ ]:
np.random.seed(2026)

# Get top 30 PCs from the standardized data
pca_30 = PCA(n_components=30, random_state=2026)
X_pca30 = pca_30.fit_transform(X_scaled)
print(f"Input to UMAP: {X_pca30.shape[0]} samples x {X_pca30.shape[1]} PCs")

# Run UMAP on the PC scores
reducer = umap.UMAP(random_state=2026)
X_umap = reducer.fit_transform(X_pca30)

# Plot UMAP embedding
fig, ax = plt.subplots(figsize=(8, 6))
for label in ['B', 'T']:
    mask = bt_type == label
    ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
               c=color_map[label], label=label, alpha=0.7,
               edgecolors='white', s=60)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('UMAP on Top 30 PCs')
ax.legend(title='Subtype')
plt.tight_layout()
plt.show()

### Exercise 4.1

Did you run UMAP on PCs or raw data? Why does this matter? (Two reasons: computational and statistical)

**YOUR ANSWER:**

---

## Part 5: UMAP on Noise

What happens when you run UMAP on pure noise? This is the critical lesson. UMAP (and t-SNE) can create apparent structure from nothing.

In [ ]:
np.random.seed(2026)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Real labels on real data
ax = axes[0]
for label in ['B', 'T']:
    mask = bt_type == label
    ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
               c=color_map[label], label=label, alpha=0.7,
               edgecolors='white', s=50)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('Real Data, Real Labels')
ax.legend(title='Subtype')

# 2. Permuted labels on real UMAP embedding
ax = axes[1]
perm_labels = bt_type.copy()
np.random.shuffle(perm_labels)
perm_colors = [color_map[t] for t in perm_labels]
for label in ['B', 'T']:
    mask = perm_labels == label
    ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
               c=color_map[label], label=label, alpha=0.7,
               edgecolors='white', s=50)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('Real Data, PERMUTED Labels')
ax.legend(title='Subtype (shuffled)')

# 3. UMAP on pure random noise (same dimensions)
ax = axes[2]
X_noise = np.random.randn(128, 30)  # same shape as PCA input
reducer_noise = umap.UMAP(random_state=2026)
X_umap_noise = reducer_noise.fit_transform(X_noise)
# Random binary labels with equal probability (emphasizing they're meaningless)
noise_labels = np.random.choice(['B', 'T'], size=128, p=[0.5, 0.5])
for label in ['B', 'T']:
    mask = noise_labels == label
    ax.scatter(X_umap_noise[mask, 0], X_umap_noise[mask, 1],
               c=color_map[label], label=label, alpha=0.7,
               edgecolors='white', s=50)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('PURE NOISE Data')
ax.legend(title='Random labels')

plt.suptitle('Can UMAP Create Structure From Nothing?', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: UMAP can create apparent clusters even from pure random noise.")
print("The clusters in the noise plot are NOT real structure.")

---

## Part 6: Parameter Sensitivity

How much do UMAP and t-SNE results depend on their tuning parameters? We test a range of `n_neighbors` (UMAP) and `perplexity` (t-SNE).

In [ ]:
# UMAP: vary n_neighbors
n_neighbors_vals = [5, 15, 30, 50]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, nn in enumerate(n_neighbors_vals):
    reducer_nn = umap.UMAP(n_neighbors=nn, random_state=2026)
    X_umap_nn = reducer_nn.fit_transform(X_pca30)
    ax = axes[i]
    for label in ['B', 'T']:
        mask = bt_type == label
        ax.scatter(X_umap_nn[mask, 0], X_umap_nn[mask, 1],
                   c=color_map[label], label=label, alpha=0.7,
                   edgecolors='white', s=40)
    ax.set_title(f'UMAP: n_neighbors = {nn}', fontsize=12)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.legend(title='Subtype', fontsize=9)

plt.suptitle('UMAP Parameter Sensitivity (n_neighbors)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# t-SNE: vary perplexity
perplexity_vals = [5, 15, 30, 50]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, perp in enumerate(perplexity_vals):
    tsne = TSNE(n_components=2, perplexity=perp, random_state=2026, max_iter=1000)
    X_tsne_perp = tsne.fit_transform(X_pca30)
    ax = axes[i]
    for label in ['B', 'T']:
        mask = bt_type == label
        ax.scatter(X_tsne_perp[mask, 0], X_tsne_perp[mask, 1],
                   c=color_map[label], label=label, alpha=0.7,
                   edgecolors='white', s=40)
    ax.set_title(f't-SNE: perplexity = {perp}', fontsize=12)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.legend(title='Subtype', fontsize=9)

plt.suptitle('t-SNE Parameter Sensitivity (perplexity)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Exercise 6.1

How does perplexity affect the t-SNE embedding? Which perplexity gives the clearest separation? How stable are the results across different seeds?

**YOUR ANSWER:**

In [ ]:
# Reproducibility test: t-SNE with different seeds vs. PCA (deterministic)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# t-SNE with seed 42
tsne_1 = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
X_tsne_1 = tsne_1.fit_transform(X_pca30)
ax = axes[0, 0]
for label in ['B', 'T']:
    mask = bt_type == label
    ax.scatter(X_tsne_1[mask, 0], X_tsne_1[mask, 1],
               c=color_map[label], label=label, alpha=0.7, edgecolors='white', s=40)
ax.set_title('t-SNE (seed=42)', fontsize=12)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(title='Subtype', fontsize=9)

# t-SNE with seed 99
tsne_2 = TSNE(n_components=2, perplexity=30, random_state=99, max_iter=1000)
X_tsne_2 = tsne_2.fit_transform(X_pca30)
ax = axes[0, 1]
for label in ['B', 'T']:
    mask = bt_type == label
    ax.scatter(X_tsne_2[mask, 0], X_tsne_2[mask, 1],
               c=color_map[label], label=label, alpha=0.7, edgecolors='white', s=40)
ax.set_title('t-SNE (seed=99)', fontsize=12)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(title='Subtype', fontsize=9)

# PCA run 1
ax = axes[1, 0]
for label in ['B', 'T']:
    mask = bt_type == label
    ax.scatter(pca_scores[mask, 0], pca_scores[mask, 1],
               c=color_map[label], label=label, alpha=0.7, edgecolors='white', s=40)
ax.set_title('PCA (run 1 -- deterministic)', fontsize=12)
ax.set_xlabel(f'PC1 ({var_explained[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_explained[1]*100:.1f}%)')
ax.legend(title='Subtype', fontsize=9)

# PCA run 2 (identical — PCA is deterministic given the same input)
pca_2 = PCA(random_state=2026)
pca_scores_2 = pca_2.fit_transform(X_scaled)
ax = axes[1, 1]
for label in ['B', 'T']:
    mask = bt_type == label
    ax.scatter(pca_scores_2[mask, 0], pca_scores_2[mask, 1],
               c=color_map[label], label=label, alpha=0.7, edgecolors='white', s=40)
ax.set_title('PCA (run 2 -- deterministic)', fontsize=12)
ax.set_xlabel(f'PC1 ({var_explained[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_explained[1]*100:.1f}%)')
ax.legend(title='Subtype', fontsize=9)

plt.suptitle('Reproducibility: t-SNE (stochastic) vs. PCA (deterministic)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("t-SNE gives DIFFERENT plots with different seeds.")
print("PCA gives the SAME plot every time.")

---

## Part 7: AI Extension

Ask an AI assistant to run t-SNE and UMAP on the ALL leukemia dataset and create a side-by-side comparison plot colored by B-cell vs. T-cell subtype. Evaluate whether the AI applies PCA first.

**Suggested prompt:**

> Run t-SNE and UMAP on the ALL leukemia dataset (12,625 genes x 128 samples) and create a side-by-side comparison plot colored by B-cell vs T-cell subtype. Use appropriate preprocessing.

In [ ]:
# YOUR CODE HERE (AI-generated or your own)


### AI Critique Checklist

| Question | Your Answer |
|----------|-------------|
| **Did the AI run PCA first?** This is THE most common error. | |
| Did it set random seeds for reproducibility? | |
| What perplexity / n_neighbors did it use? | |
| Do both methods show B/T separation? | |

---

## Method Comparison Summary

| Property | PCA | t-SNE | UMAP |
|----------|-----|-------|------|
| Linear? | Yes | No | No |
| Preserves global structure? | Yes | No | Partially |
| Preserves local structure? | Partially | Yes | Yes |
| Deterministic? | Yes | No | No |
| Interpretable axes? | Yes (loadings) | No | No |
| Needs parameter tuning? | No (beyond n_components) | Yes (perplexity) | Yes (n_neighbors, min_dist) |
| Scales to large n? | Yes | Poorly | Yes |

---

## Wrap-Up Questions

1. All three methods separate B from T. Does this mean the separation is linear or nonlinear? Or could it be either?

2. Run t-SNE three times with different seeds. Do you get the same plot? What about PCA?

3. A collaborator shows you a UMAP plot and says "look how far apart these two populations are." What do you say?

4. When would t-SNE or UMAP reveal structure that PCA misses entirely?

---

## Looking Ahead

PCA and UMAP show that B-cell and T-cell ALL are well separated. But dimension reduction is descriptive — it does not give you a predictive model or tell you which specific genes matter most. In **Lab 4**, you will use penalized regression to find a small set of genes that can *predict* cell type, and compare that gene list to the FDR list from Lab 2.